# Limit Order Book

`LimitOrderBook.jl` is a matching engine: it stores resting limit orders by price and arrival time and matches incoming orders in price-time priority. This notebook first uses it by hand, then drives it with zero-intelligence order flow (Smith, Farmer, Gillemot and Krishnamurthy 2003) to see what the book mechanics alone produce.

In [ ]:
using LimitOrderBook, Random, Statistics, StatsBase, Plots

## Orders by hand

`OrderBook{Int,Int,Int,Int}` stores sizes, prices, order ids and account ids as integers. Prices are in ticks of 0.01, so 10001 means 100.01.

In [ ]:
ob = OrderBook{Int,Int,Int,Int}()
for (id, px, sz) in [(1, 9_998, 4), (2, 9_999, 2), (3, 9_999, 3)]   # two orders queue at 99.99
    submit_limit_order!(ob, id, BUY_ORDER, px, sz)
end
for (id, px, sz) in [(4, 10_001, 2), (5, 10_002, 3), (6, 10_004, 5)]
    submit_limit_order!(ob, id, SELL_ORDER, px, sz)
end
println("Best bid and ask: ", best_bid_ask(ob))
println("Shares per side:  ", volume_bid_ask(ob))
book_depth_info(ob, 3)

## Walking the book

A market order for 4 shares takes all 2 shares at 100.01 and 2 of the 3 at 100.02: its average price is worse than the best ask. This is **price impact**.

In [ ]:
fills, left = submit_market_order!(ob, BUY_ORDER, 4)
println([(o.orderid, o.price, o.size) for o in fills])
println("Average price: ", sum(o.price * o.size for o in fills) / sum(o.size for o in fills) / 100)
println("New best bid and ask: ", best_bid_ask(ob))

At the bid, orders 2 and 3 share the price 99.99, and order 2 arrived first, so a sell market order fills it first (**time priority**). Cancelling needs the order id, its side and its price.

In [ ]:
fills, _ = submit_market_order!(ob, SELL_ORDER, 3)
println([(o.orderid, o.size) for o in fills])   # order 2 in full, then 1 share of order 3
cancel_order!(ob, 1, BUY_ORDER, 9_998)
n_orders_bid_ask(ob)

## Zero-intelligence order flow

Orders now arrive at random. Limit orders arrive at rate $\alpha$ per price level over $K$ levels, at a uniform distance from the opposite quote and never crossing it. Market orders arrive at rate $\mu$, and each resting order is cancelled at rate $\delta$. Each event is drawn in proportion to its rate.

In [ ]:
function zi(; N = 100, T = 5000, F = 100.0, alpha = 0.5, mu = 2.0, delta = 0.02, K = 20, seed = 1)
    rng, tick = Xoshiro(seed), 0.01
    ob = OrderBook{Int,Int,Int,Int}()
    live = Dict{Int,Tuple{OrderSide,Int}}()
    m0 = round(Int, F / tick)
    for id in 1:N   # seed the book around F
        side = isodd(id) ? BUY_ORDER : SELL_ORDER
        px = side == BUY_ORDER ? m0 - rand(rng, 1:K) : m0 + rand(rng, 1:K)
        submit_limit_order!(ob, id, side, px, 1)
        live[id] = (side, px)
    end
    tp, spread, id = Float64[], fill(NaN, T), N
    for t in 1:T
        bid, ask = best_bid_ask(ob)
        ref_bid = something(bid, something(ask, m0 + 1) - 1)
        ref_ask = something(ask, ref_bid + 1)
        rates = cumsum([alpha * K, alpha * K, mu / 2, mu / 2, delta * length(live)])
        e = findfirst(>=(rand(rng) * rates[end]), rates)
        if e <= 2
            id += 1
            side = e == 1 ? BUY_ORDER : SELL_ORDER
            px = e == 1 ? max(1, ref_ask - rand(rng, 1:K)) : ref_bid + rand(rng, 1:K)
            submit_limit_order!(ob, id, side, px, 1)
            live[id] = (side, px)
        elseif e <= 4
            fills, _ = submit_market_order!(ob, e == 3 ? BUY_ORDER : SELL_ORDER, 1)
            foreach(o -> delete!(live, o.orderid), fills)
            isempty(fills) || push!(tp, fills[1].price * tick)
        else
            cid = rand(rng, collect(keys(live)))
            cancel_order!(ob, cid, live[cid]...)
            delete!(live, cid)
        end
        bid, ask = best_bid_ask(ob)
        isnothing(bid) || isnothing(ask) || (spread[t] = (ask - bid) * tick)
    end
    return (; tp, spread, ob)
end

z = zi()
println("Trades: ", length(z.tp), ", mean spread: ", round(mean(filter(!isnan, z.spread)); digits = 3))
plot(z.tp; label = "Trade Price", xlabel = "Trade", ylabel = "Price")

## Market orders eat liquidity

Double the market-order rate and compare the spread and the number of resting orders over the same 20 seeds.

In [ ]:
stats(s) = (spread = mean(filter(!isnan, s.spread)), depth = sum(n_orders_bid_ask(s.ob)))
base = [stats(zi(seed = s)) for s in 1:20]
fast = [stats(zi(mu = 4.0, seed = s)) for s in 1:20]
for k in (:spread, :depth)
    println(rpad(string(k), 7), "mu = 2: ", round(mean(getindex.(base, k)); sigdigits = 3), "   mu = 4: ", round(mean(getindex.(fast, k)); sigdigits = 3))
end

## Exercises

1. Set the cancellation rate `delta = 0`. Why does the book keep growing?
2. Plot the cumulative depth `cumsum(book_depth_info(z.ob, 10)[:ASK][:volume])` against price. Is it concave or convex?
3. Submit a 2-share **sell** limit order at a price below the best bid with `submit_limit_order!` and inspect what it returns. Compare with the Known Issues table on the Limit Order Book wiki page.
4. Compute the excess kurtosis of `diff(log.(z.tp))`. Does zero-intelligence flow produce fat tails?